# Повтор: сохранённая модель и новый выпуск
Первый проект заморожен отдельно. Здесь его копия: прежний рецепт no-weather/trend_hgb, без повторного подбора. Протокол нового выпуска записан до fit: 1сентября2025, горизонт сентябрь–ноябрь, последние наблюдения31августа. Это адресный повтор знакомого материала.


In [1]:
import os,sys,subprocess,json,time
from pathlib import Path
for name in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']: os.environ[name]='1'
import numpy as np,pandas as pd,sklearn
import run as project
from IPython.display import display
from sklearn.metrics import mean_squared_error,mean_absolute_error
RANDOM_STATE=11
print('Python',sys.version.split()[0],'numpy',np.__version__,'pandas',pd.__version__,'sklearn',sklearn.__version__)
def command(*args):
 result=subprocess.run([sys.executable,'run.py',*args],check=True,text=True,capture_output=True)
 print(result.stdout)


Python 3.12.14 numpy 2.4.6 pandas 3.0.6 sklearn 1.8.0


## Цель и данные
Повторить июльский план из сохранённой модели без обучения и выпустить новый сентябрьский план через открытый параметр даты. Данные — учебная дневная выручка6кофеен. История ограничивается до выпуска; погода исключена. Календарь будущих акций предполагается утверждённым, реального журнала утверждения в CSV нет.


In [2]:
source=pd.read_csv('data/daily_sales.csv',parse_dates=['date'])
print(source.shape,source.date.min(),source.date.max())
print('duplicates',source.duplicated(['date','shop_id']).sum(),'missing',source.isna().sum().sum())
print(Path('protocol-before-fit.json').read_text())


(4386, 9) 2024-01-01 00:00:00 2025-12-31 00:00:00
duplicates 0 missing 0
{
  "as_of": "2025-09-01",
  "last_allowed_observation": "2025-08-31",
  "horizon": [
    "2025-09-01",
    "2025-11-30"
  ],
  "recipe": "Previously selected trend_hgb, no weather, 200 iterations, depth3, lr.05, seed11",
  "selection": "No re-selection; no change after viewing new horizon outcomes",
  "available_inputs": [
    "completed sales before as_of",
    "request calendar assumed approved at as_of"
  ],
  "invariance": "Change future revenue and weather only, preserve calendar, compare issued predictions",
  "key": [
    "date",
    "shop_id"
  ],
  "units": "thousand RUB per shop-day"
}


**Вывод по данным.** Ключи дат и кофеен проверяются до запуска. Старый июльский план и новый сентябрьский запрос — разные выпуски. Для обоих сценарий погоды не требуется, так как этот собственный рецепт не использует погоду. Дата отсчёта и категории входят в сохранённую подготовку признаков.


## Анализ и модели
Сначала saved выполняет только predict. Затем issue пересобирает историю до открытого as_of и заново fit по прежнему рецепту. Проверка меняет будущие выручку и погоду при неизменном календаре; прогноз должен сохраниться.


In [3]:
command('saved','--model','model.joblib','--metadata','metadata.json','--request','request.csv','--output','reproduced-plan.csv')
a=pd.read_csv('plan.csv');b=pd.read_csv('reproduced-plan.csv')
joined=a.merge(b,on=['as_of','date','shop_id'],suffixes=('_old','_new'),validate='one_to_one')
assert len(joined)==len(a)==len(b) and np.allclose(joined.prediction_k_old,joined.prediction_k_new,atol=1e-9,rtol=0)
print('saved model agreement:',len(joined),'keys')


saved model: no fit; rows 1104 as_of 2025-07-01

saved model agreement: 1104 keys


In [4]:
command('issue','--as-of','2025-09-01','--history','data/daily_sales.csv','--request','new-request.csv','--output','new-plan.csv','--model-out','new-issue.joblib','--metadata-out','new-issue.json','--verify-future-invariance')
command('saved','--model','new-issue.joblib','--metadata','new-issue.json','--request','new-request.csv','--output','new-reproduced.csv')
a=pd.read_csv('new-plan.csv');b=pd.read_csv('new-reproduced.csv')
joined=a.merge(b,on=['as_of','date','shop_id'],suffixes=('_issued','_reloaded'),validate='one_to_one')
assert len(joined)==len(a)==len(b) and np.allclose(joined.prediction_k_issued,joined.prediction_k_reloaded,atol=1e-9,rtol=0)
print('new issue saved model agreement:',len(joined),'keys')


new issue: fixed recipe 2025-09-01 rows 546 history through 2025-08-31
{
  "as_of": "2025-09-01",
  "last_available_observation": "2025-08-31",
  "history_rows": 3654,
  "horizon_start": "2025-09-01",
  "horizon_end": "2025-11-30",
  "recipe": {
    "name": "trend_hgb",
    "max_iter": 200,
    "max_depth": 3,
    "learning_rate": 0.05,
    "random_state": 11,
    "weather": "excluded",
    "origin": "2024-01-01"
  },
  "selection": "Fixed previous own recipe; no new selection on final period",
  "weather": "Excluded; no scenario or future weather consumed",
  "calendar": "Request holiday/promo assumed approved at issue; source has no approval log",
  "keys": [
    "date",
    "shop_id"
  ],
  "units": "thousand RUB per shop-day",
  "fit_s": 0.06773875001817942,
  "predict_ms": 4.962541977874935,
  "versions": {
    "python": "3.12.14",
    "numpy": "2.4.6",
    "pandas": "3.0.6",
    "sklearn": "1.8.0"
  },
  "threads": 1,
  "future_fact_mutation_check": "passed: future revenue/weathe

saved model: no fit; rows 546 as_of 2025-09-01

new issue saved model agreement: 546 keys


In [5]:
issued=pd.read_csv('new-plan.csv',parse_dates=['date'])
eval_rows=issued.merge(source[['date','shop_id','revenue_k']],on=['date','shop_id'],validate='one_to_one')
history=source[source.date<'2025-09-01'];categories={c:sorted(history[c].unique()) for c in project.CATS}
request=pd.read_csv('new-request.csv',parse_dates=['date'])
base=project.fit('frozen_week',history,categories)
base_pred=project.predict(base,request)
base_rows=request[['date','shop_id']].assign(baseline_k=base_pred)
eval_rows=eval_rows.merge(base_rows,on=['date','shop_id'],validate='one_to_one')
metrics={name:{'rmse_k':float(np.sqrt(mean_squared_error(eval_rows.revenue_k,eval_rows[col]))),'mae_k':float(mean_absolute_error(eval_rows.revenue_k,eval_rows[col]))} for name,col in [('fixed_trend_hgb','prediction_k'),('frozen_week','baseline_k')]}
pd.DataFrame(metrics).T.to_csv('new-metrics.csv')
Path('versions.json').write_text(json.dumps(json.loads(Path('new-issue.json').read_text())['versions'],indent=2))
display(pd.DataFrame(metrics).T)
print('new plan as_of values',issued.as_of.unique(),'rows',len(issued))


,rmse_k,mae_k
fixed_trend_hgb,7.855623,5.814838
frozen_week,14.418980,10.308791


new plan as_of values <ArrowStringArray>
['2025-09-01']
Length: 1, dtype: str rows 546


**Вывод по анализу.** Выпуск1сентября использует только завершённую историю по31августа. Рецепт200итераций/depth3/lr.05/seed11 фиксирован до расчёта; новые метрики служат отчёту по этому знакомому периоду и не меняют выбор. Согласованность сохранённой модели, ключей и подмены будущих фактов проверяется assert, результаты сохраняются в metadata.


## Выводы и рекомендации
Новый выпуск 1 сентября 2025 рассчитан на сентябрь–ноябрь: 546 строк, история по 31 августа. Фиксированный trend_hgb дал RMSE 7.856 тыс. ₽ и MAE 5.815 тыс. ₽ против RMSE 14.419 и MAE 10.309 у замороженного недельного ориентира на тех же датах и кофейнях.
Saved-режим повторил 1104 июльских ключа без fit; новый сохранённый выпуск повторил все 546 ключей. Подмена будущей выручки и погоды при неизменном календаре сохранила прогнозы. Даты выпуска включены в plan и metadata. Рецепт после оценки не менялся.
Предлагаю пилот решений о сменах и закупках относительно действующего правила. Нужно отдельно собрать стоимость лишних смен, нехватку персонала и потерянное обслуживание, списания и наличие молока. Ошибка выручки в тыс. ₽ не равна этим затратам. Календарь акций считается утверждённым, погода исключена; перенос на другой бизнес не доказан.
